# export

In [ ]:
import sys
import torch


sys.path.insert(0, "darya-tts/")
from inference_utils_darya import *
sys.path.append("/home/ubuntu/zs_cleaning/dune_codec")

CONFIG_PATH = "darya-tts/config_transformer.json"
CKPT_DIR    = "darya-tts/exp/darya_fix_2ndstage/checkpoints/step_0070000"

tts_model, tts_cfg = load_model_from_checkpoint(CONFIG_PATH, CKPT_DIR, use_speaker_conditioning=True) # or false, if you're using the vanilla checkpoint


In [ ]:
import sys
import torch

from nb_export_tts_core_full_onnx import export_tts_core_full_onnx_from_loaded_model

tts_model = tts_model.to("cpu", dtype=torch.bfloat16).eval()

EXPORT_BF16_VALUED_ONNX = export_tts_core_full_onnx_from_loaded_model(
    model=tts_model,
    cfg=tts_cfg,
    out_dir="alch_onnx_fp32",
    device="cpu",
    precision="fp32",
    use_speaker_conditioning=True,
    dummy_batch=1,
    dummy_text_len=86,
    dummy_audio_len=192,
    opset=18,
    exporter="dynamo",
    run_onnx_checker=True,
)



## 8bit

In [ ]:
from pathlib import Path
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

FP32_ONNX_DIR = Path("darya-tts/ONNX/alch_onnx_fp32").resolve()
INT8_ONNX_DIR = Path("/darya-tts/ONNX/alch_onnx_int8").resolve()
TMP_ONNX_DIR = INT8_ONNX_DIR / "_sanitized"

INT8_ONNX_DIR.mkdir(parents=True, exist_ok=True)
TMP_ONNX_DIR.mkdir(parents=True, exist_ok=True)

ENC_FP32 = FP32_ONNX_DIR / "tts_encoder_fp32.onnx"
DEC_FP32 = FP32_ONNX_DIR / "tts_decoder_denoiser_fp32.onnx"

ENC_SAN = TMP_ONNX_DIR / "tts_encoder_fp32_sanitized.onnx"
DEC_SAN = TMP_ONNX_DIR / "tts_decoder_denoiser_fp32_sanitized.onnx"

ENC_INT8 = INT8_ONNX_DIR / "tts_encoder_int8.onnx"
DEC_INT8 = INT8_ONNX_DIR / "tts_decoder_denoiser_int8.onnx"


def sanitize_onnx(src_path, dst_path):
    model = onnx.load(str(src_path), load_external_data=True)
    del model.graph.value_info[:]
    onnx.save_model(
        model,
        str(dst_path),
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location=dst_path.name + ".data",
        size_threshold=1024,
        convert_attribute=False,
    )
    onnx.checker.check_model(str(dst_path))
    return dst_path


sanitize_onnx(ENC_FP32, ENC_SAN)
sanitize_onnx(DEC_FP32, DEC_SAN)

quantize_dynamic(
    model_input=str(ENC_SAN),
    model_output=str(ENC_INT8),
    op_types_to_quantize=["MatMul"],
    weight_type=QuantType.QInt8,
    per_channel=False,
    reduce_range=True,
    use_external_data_format=True,
    extra_options={
        "MatMulConstBOnly": True,
    },
)

quantize_dynamic(
    model_input=str(DEC_SAN),
    model_output=str(DEC_INT8),
    op_types_to_quantize=["MatMul"],
    weight_type=QuantType.QInt8,
    per_channel=False,
    reduce_range=True,
    use_external_data_format=True,
    extra_options={
        "MatMulConstBOnly": True,
    },
)

onnx.checker.check_model(str(ENC_INT8))
onnx.checker.check_model(str(DEC_INT8))

print("encoder:", ENC_INT8)
print("decoder:", DEC_INT8)

# inference

In [ ]:
import sys
import time
sys.path.insert(0,"/PATH/TO/dune_codec")
sys.path.insert(1,"darya-tts")


from Text_Preprocessors.TextPreprocessor import TextPreprocessor

from inference_utils_darya import *
from inference_utils import *


from transformers import AutoTokenizer
tokenizer_path = "KavirLabs/Darya_Tokenizer_bpe"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, add_bos_token=True, add_eos_token=True)

device = "cpu"

from codec.audio_processing.dune_codec import load_dune_audio_tokenizer
dac_model = load_dune_audio_tokenizer("Respair/dune_codec", 
                                         device=device)


duration_model = load_duration_model(
    "duration_predictor/dur/exp/ckpts_02/multilingual_ce_fixed/checkpoints/step_0075000.pt",
    vocab_size=4096,
    latent_dim=52,
    hidden_dim=256,
    n_text_layer=8,
    n_cross_layer=8,
    n_head=8,
    output_dim=378,
    device=device,
)

FP32_ONNX_DIR = Path("darya-tts/ONNX/onnx_fp32").resolve()

ENC_FP32 = "onnx_fp32/tts_encoder_fp32.onnx"
DEC_FP32 = "onnx_int8/tts_decoder_denoiser_int8.onnx"


onnx_core = DaryaONNXCore(
    encoder_path=ENC_FP32,
    decoder_path=DEC_FP32,
    provider="CPUExecutionProvider",
)



TE = TextPreprocessor.load(
    mode="all",  # "all", "persian", or "russian" english uses graphemes, so no conversion is needed. "all" derives it automatically
    persian_model_path="Text_Preprocessors/Finglish/persian_transliterator-q8_0.gguf",
    ruphon_workdir="./models",
    device="CPU",
    n_ctx=768,
    n_threads=8,
)



In [ ]:
import time
import re
import torch
import numpy as np


def strip_tags(text: str) -> str:
    return re.sub(r'^(?:<[^>]+>\s*)+', '', text).strip()


text = """<S1><persian> من در وقت تنفس وقتی آیت‌الله خامنه‌ای را دیدم، بیشتر بحث شورای رهبری مطرح بود. ایشان به من گفت که ما باید برسیم به شورا و حتی اعضای شورا را هم با ایشان مرور کردیم. حتی اینکه سه نفر باشند، یا ۵ نفر و اسامی چند نفر هم مطرح شد. من خدمت آقا گفتم که اگر شورایی شد، شورای سه نفره به مراتب بهتر از پنج نفره است. هرچه کوچکتر، تصمیم‌گیری راحت‌تر خواهد بود."""



text = TE.process(text)

if "<persian>" in text:
    text = text.replace("e and.", "and.")
    
print(text)

td = Extractor(
    tokenizer=tokenizer,
    device=device,
    duration_model=duration_model,
    speaker_model=None,
)


duration_text = strip_tags(text.replace(".", ".")).strip()

if duration_text.endswith(","):
    duration_text = duration_text[:-1] + "."

text_ids, text_mask, duration_text_ids, duration_text_mask = td.get_tokens(
    text,
    duration_text,
)

speed = 1.
n_frame_per_class = 1
latent_size = 52
max_duration = int(12.5 * 30)

duration = td.get_duration(
                        duration_text_ids,
                        duration_text_mask,
                        speed=speed,
                        n_frame_per_class=n_frame_per_class,
                        min_total_frames=1,
                        max_total_frames=max_duration,
                    )


print(duration.float() / 12.5)


if device == "cuda":
    torch.cuda.synchronize()

t0 = time.perf_counter()

output = sample_euler_reducio_onnx(
                    core=onnx_core,
                    text_ids=text_ids,
                    text_mask=text_mask,
                    latent_size=latent_size,
                    duration=duration,
                    cond_latents=None,
                    cond_latent_mask=None,
                    steps=64,
                    cfg=3.0,
                    cfg_min_t=0.0,
                    cfg_max_t=0.85,
                    apg_eta=0.0,
                    apg_momentum=0.0,
                    apg_norm=None,
                    seed=None,
                    torch_output_device=device,
                )

if device == "cuda":
    torch.cuda.synchronize()

infer_time = time.perf_counter() - t0

audio_cropped = decode_audio(
    dac_model,
    output.transpose(1, 2),
)

if device == "cuda":
    torch.cuda.synchronize()

total_time = time.perf_counter() - t0

audio_duration = audio_cropped.shape[-1] / 44100
rtf_infer_only = infer_time / max(audio_duration, 1e-6)
rtf_total = total_time / max(audio_duration, 1e-6)

print(
    f"used_frames={int(duration[0].item())} | "
    f"infer={infer_time:.3f}s | "
    f"total={total_time:.3f}s | "
    f"audio={audio_duration:.2f}s | "
    f"rtf_infer={rtf_infer_only:.4f} | "
    f"rtf_total={rtf_total:.4f}"
)

Sawt(audio_cropped, rate=44100)

<S1><persian> man dar vaqte tanaffos vaqti AyatollAh khAmene 'i rA didam, bishtar bahse shorAye rahbari matrah bud. ishAn be man goft ke mA bAyad beresim be shorA va hattA a'zAye shorA rA ham bA ishAn morur kardim. hattA inke se nafar bAshand, yA panj nafar va asAmiye chand nafar ham matrah shod. man khedmate AqA goftam ke agar shorAi shod, shorAye se nafare be marAteb behtar az panj nafare ast. harche kuchektar, tasmimgiri rAhat tar khAhad bud.
tensor([29.6000])
used_frames=370 | infer=16.108s | total=17.518s | audio=29.60s | rtf_infer=0.5442 | rtf_total=0.5918


## prompt

In [ ]:
AUDIO_PATH = "path"
import librosa


wav = librosa.load(AUDIO_PATH, sr=22050, duration=30)[0]

prompt_tensor = torch.from_numpy(wav)
# prompt_text = decode_ids(beam_text[0])
prompt_text = "<S1><en> There is nothing, absolutely nothing you can do to change this situation!"
print(prompt_text)



@torch.no_grad()
def extract_prequant_latents(dac_model, wav: torch.Tensor, device="cuda"):
    wav = wav.float()

    audio = wav.unsqueeze(0).to(device)
    audio_len = torch.tensor([wav.shape[0]], dtype=torch.long, device=device)

    encoder = dac_model.codec.audio_encoder
    pre_q, latent_len = encoder(audio=audio, audio_len=audio_len)

    pre_q = pre_q.transpose(1, 2)

    valid_len = int(latent_len[0].item())
    pre_q = pre_q[:, :valid_len, :]

    print("fixed pre_q:", pre_q.shape)

    return pre_q[0]

prompt_lats = extract_prequant_latents(dac_model, prompt_tensor, device=device)
Sawt(wav, rate=22050)

In [ ]:
import time
import re
import torch
import numpy as np


def strip_tags_for_duration(text: str) -> str:
    return re.sub(r"<[^>]+>\s*", "", text).strip()


text = """<S1><en> Wait, so, uh, you actually told him you weren't coming? <S2> Well, I mean, I didn't say it like that. <S1> What did you say? <S2> I said, "I don't think I'll make it," which is, you know, basically the same thing. <S1> No, it's not. That's the kind of message that makes people wait around for two hours. <S2> Okay, yeah, fair, but I thought he'd get what I meant."""

td = Extractor(
    tokenizer=tokenizer,
    device=device,
    duration_model=duration_model,
    speaker_model=None,
)


full_text = prompt_text + " " + text
duration_text = strip_tags_for_duration(full_text)

text_ids, text_mask, duration_text_ids, duration_text_mask = td.get_tokens(
    full_text,
    duration_text,
)

prompt_lats_b = prompt_lats.unsqueeze(0)

prompt_mask = torch.ones(
    1,
    prompt_lats.shape[0],
    device=device,
    dtype=torch.bool,
)

speed = 1.0
n_frame_per_class = 1
latent_size = 52
max_duration = int(12.5 * 30)

T_prompt = prompt_lats.shape[0]


duration = td.get_duration(
                duration_text_ids,
                duration_text_mask,
                speed=speed,
                cond_latents=prompt_lats_b,
                cond_latent_mask=prompt_mask,
                n_frame_per_class=n_frame_per_class,
                min_total_frames=T_prompt + 1,
                max_total_frames=max_duration,
            )

if device == "cuda":
    torch.cuda.synchronize()

t0 = time.perf_counter()

output = sample_euler_reducio_onnx( core=onnx_core, text_ids=text_ids, text_mask=text_mask, 
                                   latent_size=latent_size, duration=duration, cond_latents=prompt_lats_b, cond_latent_mask=prompt_mask,
                                    steps=30,
                                    cfg=2.0,
                                    seed=None,
                                    torch_output_device=device,
                                )

if device == "cuda":
    torch.cuda.synchronize()

infer_time = time.perf_counter() - t0

continuation_lats = output[:, T_prompt:int(duration[0].item()), :]

audio_cropped = decode_audio(
    dac_model,
    continuation_lats.transpose(1, 2),
)

if device == "cuda":
    torch.cuda.synchronize()

total_time = time.perf_counter() - t0

audio_duration = audio_cropped.shape[-1] / 44100
rtf_infer_only = infer_time / max(audio_duration, 1e-6)
rtf_total = total_time / max(audio_duration, 1e-6)

print(
    f"used_frames={int(duration[0].item())} | "
    f"prompt_frames={T_prompt} | "
    f"generated_frames={continuation_lats.shape[1]} | "
    f"infer={infer_time:.3f}s | "
    f"total={total_time:.3f}s | "
    f"audio={audio_duration:.2f}s | "
    f"rtf_infer={rtf_infer_only:.4f} | "
    f"rtf_total={rtf_total:.4f}"
)

Sawt(audio_cropped, rate=44100)

# 🥣

In [ ]:
import sys
import time

from Text_Preprocessors.TextPreprocessor import TextPreprocessor

from inference_utils_darya import *
from inference_utils import *
from transformers import AutoTokenizer

tokenizer_path = "KavirLabs/Darya_Tokenizer_bpe"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, add_bos_token=True, add_eos_token=True)


ENC_FP32 = "darya-tts/ONNX/alch_onnx_fp32/tts_encoder_fp32.onnx"
DEC_FP32 = "darya-tts/ONNX/alch_onnx_int8/tts_decoder_denoiser_int8.onnx"


onnx_core = DaryaONNXCore(
    encoder_path=ENC_FP32,
    decoder_path=DEC_FP32,
    provider="CPUExecutionProvider",
)

import nemo.collections.asr as nemo_asr
speaker_model = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained("nvidia/speakerverification_en_titanet_large")

In [ ]:
import time
import re
import torch
import numpy as np


def strip_tags(text: str) -> str:
    return re.sub(r'^(?:<[^>]+>\s*)+', '', text).strip()


text = """<S1><en> Some things benefit from shocks; they thrive and grow when exposed to volatility, randomness, disorder, and stressors and love adventure, risk, and uncertainty. yet, in spite of the ubiquity of the phenomenon, there is no word for the exact opposite of fragile. Let us call it antifragile."""

text = TE.process(text)

if "<persian>" in text:
    text = text.replace("e and.", "and.")
    
print(text)

td = Extractor(
    tokenizer=tokenizer,
    device=device,
    duration_model=duration_model,
    speaker_model=speaker_model,
)


duration_text = strip_tags(text.replace(".", ",")).strip()

if duration_text.endswith(","):
    duration_text = duration_text[:-1] + "."

text_ids, text_mask, duration_text_ids, duration_text_mask = td.get_tokens(
    text,
    duration_text,
)


emb = td.get_speaker_embedding("darya-tts/ONNX/generated_simple__95c233828484173c__1783547625055_85983cb5.wav")

speed = 1.
n_frame_per_class = 1
latent_size = 52
max_duration = int(12.5 * 30)

duration = td.get_duration(
    duration_text_ids,
    duration_text_mask,
    speed=speed,
    n_frame_per_class=n_frame_per_class,
    min_total_frames=1,
    max_total_frames=max_duration,
)


print(duration.float() / 12.5)

### SPEECH EDIT 

In [ ]:
AUDIO_PATH = "darya-tts/multi.wav"
import librosa


wav , sr = librosa.load(AUDIO_PATH, sr=22050)

prompt_tensor = torch.from_numpy(wav)

prompt_lats = extract_prequant_latents(dac_model, prompt_tensor, device=device)
Sawt(wav[:], rate=sr)

In [ ]:
AUDIO_TO_EDIT = AUDIO_PATH

target_text = "<S1><LANGUAGE> your text + the part you want to edit"

wav, wav_sr = librosa.load(AUDIO_TO_EDIT, sr=22050)
wav_tensor = torch.from_numpy(wav)

original_lats = extract_prequant_latents(dac_model, wav_tensor, device=device)

codec_rate_hz = 12.5

audio_len = wav.shape[-1] / 22050
# Loose edit regions.

# if you only mask the first one, the second changed text will fight against preserved original audio
parts_to_edit = [
    [11, 13],
]


# positive value = add time
# 0 = use original duration baseline
# negativr value = shorten.

extra_duration = [
    0.1,   # add some room 
]

cond, keep_mask, span_mask, valid, edit_ranges = build_latent_edit_condition_onnx(
    original_lats,
    wav_num_samples=len(wav),
    wav_sr=wav_sr,
    parts_to_edit=parts_to_edit,
    extra_duration=extra_duration,
    duration_scale=0.8,
    padding_sec=0.5,
    min_edit_sec=0.15,
    codec_rate_hz=codec_rate_hz,
)

td = Extractor(tokenizer, device, duration_model, None)

duration_text = re.sub(r"<[^>]+>\s*", "", target_text).strip()

text_ids, text_mask, _, _ = td.get_tokens(
    target_text,
    duration_text,
)

edited_lats = sample_euler_reducio_edit_onnx(
    onnx_core,
    text_ids=text_ids,
    text_mask=text_mask,
    cond=cond,
    keep_mask=keep_mask,
    valid_mask=valid,
    steps=8,
    cfg=2.,
    seed=None,
)

audio_edited = decode_audio(
    dac_model,
    edited_lats.transpose(1, 2),
)

Sawt(audio_edited, rate=44100)

fixed pre_q: torch.Size([1, 158, 52])
